#  **07 - Lap Time Prediction Model**

## **Objective**
Build a forward-looking lap time prediction model. Given only information known 
before a lap starts (driver, tyre state, weather conditions, track status, 
circuit), predict the lap time. This model is meant to power strategy 
simulation features in the web app ("what-if" scenarios), so it deliberately 
excludes any in-lap telemetry or post-lap flags.

## **Design decision**
Two options were considered:
1. Descriptive model - use full lap telemetry (sector times, avg/max speed, 
   throttle, etc) to predict LapTime. High accuracy, but useless for a live 
   simulation since none of those values exist before the lap is driven.
2. Forward-looking model - use only pre-lap known info (tyre age, compound, 
   weather at lap start, track status, driver, circuit) to predict LapTime 
   before it happens.

Going with option 2. Expected tradeoff: lower R2 / higher error compared to a 
descriptive model, since the most predictive features (which are computed 
during or after the lap) are intentionally excluded. This is the honest 
ceiling of the problem being solved, not a modeling failure.

## **Leakage rule for this notebook**
A feature is allowed only if it would be known in reality at the moment 
before the driver starts this lap. This will be checked column by column 
against the 78 columns in fastf1_ml_base.parquet before finalizing the 
feature list.

## **Source data**
data/processed/fastf1_ml_base.parquet - 469,012 rows x 78 columns

In [1]:
import pandas as pd
import numpy as np

df = pd.read_parquet("../data/processed/fastf1_ml_base.parquet")

print(df.shape)
df.head()

(469012, 78)


,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,Compound_category,status_green,status_yellow,status_unused3,status_safety_car,status_red_flag,status_vsc,status_vsc_ending,status_unknown,LapTime_seconds
0,0 days 00:16:44.279000,RAI,7,NaT,1.0,1.0,0 days 00:14:32.810000,0 days 00:16:43.205000,NaT,0 days 00:00:53.588000,...,SOFT,True,False,False,False,False,False,False,False,NaN
1,0 days 00:16:59.665000,ERI,9,NaT,1.0,1.0,0 days 00:14:36.454000,0 days 00:16:58.524000,NaT,0 days 00:01:02.254000,...,SOFT,True,False,False,False,False,False,False,False,NaN
2,0 days 00:17:00.879000,VAN,2,NaT,1.0,1.0,0 days 00:14:58.922000,0 days 00:16:59.711000,NaT,0 days 00:00:49.023000,...,SOFT,True,False,False,False,False,False,False,False,NaN
3,0 days 00:17:07.203000,KUB,40,NaT,1.0,1.0,0 days 00:15:04.381000,NaT,NaT,0 days 00:00:51.612000,...,HARD,True,False,False,False,False,False,False,False,NaN
4,0 days 00:17:29.430000,SAI,55,NaT,1.0,1.0,0 days 00:15:09.463000,0 days 00:17:27.869000,NaT,0 days 00:00:57.787000,...,MEDIUM,True,False,False,False,False,False,False,False,NaN


In [2]:
print(df.columns.tolist())

['Time', 'Driver', 'DriverNumber', 'LapTime', 'LapNumber', 'Stint', 'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime', 'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest', 'Compound', 'TyreLife', 'FreshTyre', 'Team', 'LapStartTime', 'LapStartDate', 'TrackStatus', 'Position', 'Deleted', 'DeletedReason', 'FastF1Generated', 'IsAccurate', 'BroadcastName', 'Abbreviation', 'DriverId', 'TeamColor', 'TeamId', 'FirstName', 'LastName', 'FullName', 'HeadshotUrl', 'CountryCode', 'FinalPosition', 'ClassifiedPosition', 'GridPosition', 'Q1', 'Q2', 'Q3', 'ResultTime', 'Status', 'Points', 'Laps', 'AirTemp', 'Humidity', 'Pressure', 'Rainfall', 'TrackTemp', 'WindDirection', 'WindSpeed', 'avg_speed', 'max_speed', 'avg_throttle', 'full_throttle_pct', 'max_rpm', 'brake_sample_pct', 'drs_active_pct', 'Season', 'EventName', 'SessionName', 'Compound_category', 'status_green', 'status_yellow', 'status_unused3', 'stat

### **IN (pre-lap known)**

 - Driver, DriverNumber, DriverId, Team, TeamId, CountryCode — driver/team identity, static
 - LapNumber, Stint (known counters, don't change mid-lap)
 - Compound, Compound_category, TyreLife, FreshTyre (tyre state at lap start)
 - GridPosition, Q1, Q2, Q3 — pre-race known (qualifying happens before race)
 - AirTemp, Humidity, Pressure, Rainfall, TrackTemp, WindDirection, WindSpeed — legit as discussed, as long as it's the reading at/before lap start
 - Season, EventName, SessionName — context identifiers

### **OUT (leakage, in-lap or post-lap)**

 - Sector1Time, Sector2Time, Sector3Time, Sector1SessionTime, Sector2SessionTime, Sector3SessionTime (in-lap telemetry)
 - SpeedI1, SpeedI2, SpeedFL, SpeedST, avg_speed, max_speed, avg_throttle, full_throttle_pct, max_rpm, brake_sample_pct, drs_active_pct — in-    lap telemetry
 - IsPersonalBest, Deleted, DeletedReason — post-laptime flags
 - PitInTime  (whether/when driver pits this lap is a decision + outcome, not known beforehand)
 - Position — this is position during/after the lap, not before
 - FinalPosition, ClassifiedPosition, ResultTime, Status, Points, Laps — race-end result fields, huge leakage
 - LapTime, LapTime_seconds (these are the target, obviously excluded from features)

### **METADATA (identifiers/junk, not real features)**

 - BroadcastName, Abbreviation, FirstName, LastName, FullName, HeadshotUrl, TeamColor  (redundant with Driver/Team or useless for modeling)
 - FastF1Generated, IsAccurate — data quality flags, not lap conditions. IsAccurate might matter as a filter for which rows to train on, not as a feature.

### **TO BE DISCUSSED**

 - PitOutTime, TrackStatus + 8 status flag, LapStartTime, LapStartDate, TyreLife(probably IN)

___
___

### **Scope to Race sessions only**
Bcs Different sessions have fundamentally different purpose with different driving style, for example - In Practise sessions, Drivers do experimental driving like Race simulation on high fuel, tyre testing, exploring track limits.

Moreover, Qualifying race fundamentally follows different physics like flash out push with low fuel,Tyre life are also mostly fresh.
___

In [3]:
df_race = df[df['SessionName'] == 'Race'].copy()
print(df_race.shape)
df_race['Season'].value_counts().sort_index()

(188482, 78)


Season
2018    21410
2019    23676
2020    18350
2021    23756
2022    23577
2023    24420
2024    26604
2025    26689
Name: count, dtype: int64

### **Derive `is_out_lap`**
if `PitOutTime` is not null , means this lap is an Out-Lap.
___

In [4]:
df_race['is_out_lap'] = df_race['PitOutTime'].notna() & (df_race['LapNumber'] != 1)
df_race['is_out_lap'].value_counts()

is_out_lap
False    182572
True       5910
Name: count, dtype: int64

### **`track_status_at_lap_start` shift approximation**
Idea: For each driver, sort the laps by LapNumber within a session and carry over the status flags from the previous lap to the current one. Since there is no previous lap for Lap 1, we will assume a green flag (the default for a race start); this is reasonable because races almost always begin under a green flag.
___

In [5]:
import numpy as np

status_cols = ['status_green', 'status_yellow', 'status_safety_car', 
               'status_red_flag', 'status_vsc', 'status_vsc_ending', 'status_unknown']

group_keys = ['Season', 'EventName', 'SessionName', 'Driver']

df_race = df_race.sort_values(group_keys + ['LapNumber'])

for col in status_cols:
    new_col = col.replace('status_', 'prevstatus_')
    shifted = df_race.groupby(group_keys)[col].shift(1)
    
    default_value = True if col == 'status_green' else False
    df_race[new_col] = np.where(shifted.isna(), default_value, shifted).astype(bool)

df_race[[c for c in df_race.columns if c.startswith('prevstatus_')]].isna().sum()

prevstatus_green         0
prevstatus_yellow        0
prevstatus_safety_car    0
prevstatus_red_flag      0
prevstatus_vsc           0
prevstatus_vsc_ending    0
prevstatus_unknown       0
dtype: int64

### **Fuel load proxy and track evolution proxy**

As there are no dynamiac features like Fuel load or track evoluation in FastF1 raw dataset,The proxy for total race laps will be the group's max LapNumber (race distance publicly known before the race, this is a reasonable approximation, may be slightly off only in edge case red-flag-shortened races, please note)
___

In [6]:
df_race['total_laps_est'] = df_race.groupby(['Season', 'EventName'])['LapNumber'].transform('max')
df_race['laps_remaining_est'] = df_race['total_laps_est'] - df_race['LapNumber'] + 1
df_race['fuel_load_proxy'] = df_race['laps_remaining_est']  # more laps remaining = more fuel onboard

df_race['track_evolution_proxy'] = df_race['LapNumber']  # simple version, session progression

df_race[['LapNumber', 'total_laps_est', 'laps_remaining_est', 'fuel_load_proxy']].sample(10)

,LapNumber,total_laps_est,laps_remaining_est,fuel_load_proxy
314030,9.0,50.0,42.0,42.0
199305,35.0,66.0,32.0,32.0
356815,14.0,52.0,39.0,39.0
128356,20.0,44.0,25.0,25.0
293606,12.0,57.0,46.0,46.0
31096,4.0,67.0,64.0,64.0
435489,26.0,70.0,45.0,45.0
111620,7.0,66.0,60.0,60.0
218903,9.0,58.0,50.0,50.0
281000,8.0,56.0,49.0,49.0


### **Tyre degradation interaction terms**
___

In [7]:
df_race['tyrelife_squared'] = df_race['TyreLife'] ** 2
df_race['compound_tyrelife'] = df_race['Compound_category'].astype(str) + '_' + df_race['TyreLife'].astype(str)

>The last one is string interaction, we have to handle it properly at the time of encoding.

### **Final Feature list**
___

In [8]:
#These features are leaked proofs , and are verified using multiple AI agents - like Kimi.ai,Claude.

feature_cols = [
    # driver/team identity
    'Driver', 'Team',
    
    # race progression
    'LapNumber', 'Stint',
    
    # tyre state
    'Compound_category', 'TyreLife', 'FreshTyre', 'tyrelife_squared',
    
    # qualifying/grid (race-only, so always known)
    'GridPosition', 'Q1', 'Q2', 'Q3',
    
    # weather
    'AirTemp', 'Humidity', 'Pressure', 'Rainfall', 'TrackTemp', 
    'WindDirection', 'WindSpeed',
    
    # track/session context
    'EventName', 'Season',
    
    # derived
    'is_out_lap', 'fuel_load_proxy', 'track_evolution_proxy',
    
    # track status at lap start
    'prevstatus_green', 'prevstatus_yellow', 'prevstatus_safety_car',
    'prevstatus_red_flag', 'prevstatus_vsc', 'prevstatus_vsc_ending', 
    'prevstatus_unknown'
]

target_col = 'LapTime_seconds'

print(len(feature_cols), "features")
df_race[feature_cols + [target_col]].isna().sum()

31 features


Driver                        0
Team                          0
LapNumber                     0
Stint                       646
Compound_category             0
TyreLife                   1361
FreshTyre                     0
tyrelife_squared           1361
GridPosition                  0
Q1                       188482
Q2                       188482
Q3                       188482
AirTemp                    1149
Humidity                   1149
Pressure                   1149
Rainfall                   1149
TrackTemp                  1149
WindDirection              1149
WindSpeed                  1149
EventName                     0
Season                        0
is_out_lap                    0
fuel_load_proxy               0
track_evolution_proxy         0
prevstatus_green              0
prevstatus_yellow             0
prevstatus_safety_car         0
prevstatus_red_flag           0
prevstatus_vsc                0
prevstatus_vsc_ending         0
prevstatus_unknown            0
LapTime_

I am going to drop Q1,Q2, and Q3 - bcs these are present only for quali sessions.

In [9]:
feature_cols = [c for c in feature_cols if c not in ['Q1', 'Q2', 'Q3']]

before_rf = len(df_race)
df_race_clean = df_race[df_race['status_red_flag'] == False].copy()
print(f"Excluded {before_rf - len(df_race_clean)} red flag rows")

model_df = df_race_clean[feature_cols + [target_col]].copy()

before = len(model_df)
model_df = model_df.dropna()
after = len(model_df)

print(f"Dropped {before - after} rows ({(before-after)/before*100:.2f}%)")
print(model_df.shape)

Excluded 426 red flag rows
Dropped 5343 rows (2.84%)
(182713, 29)


In [10]:
model_df.dtypes

Driver                    object
Team                      object
LapNumber                float64
Stint                    float64
Compound_category         object
TyreLife                 float64
FreshTyre                   bool
tyrelife_squared         float64
GridPosition             float64
AirTemp                  float64
Humidity                 float64
Pressure                 float64
Rainfall                 float64
TrackTemp                float64
WindDirection            float64
WindSpeed                float64
EventName                 object
Season                     int64
is_out_lap                  bool
fuel_load_proxy          float64
track_evolution_proxy    float64
prevstatus_green            bool
prevstatus_yellow           bool
prevstatus_safety_car       bool
prevstatus_red_flag         bool
prevstatus_vsc              bool
prevstatus_vsc_ending       bool
prevstatus_unknown          bool
LapTime_seconds          float64
dtype: object

### **Categorical Encoding**
>As we will use Tree Based Model XGBoost, so we don't need to OHE, it handles Label Encoding well specially if features with high cardinality.

In [11]:
from sklearn.preprocessing import LabelEncoder

categorical_cols = ['Driver','Team','Compound_category','EventName']

encoders = {}
for col in categorical_cols:
    le=LabelEncoder()
    model_df[col + '_enc']= le.fit_transform(model_df[col])
    encoders[col]=le

model_df[[c+'_enc' for c in categorical_cols]].head()

,Driver_enc,Team_enc,Compound_category_enc,EventName_enc
1709,2,9,2,1
1790,2,9,2,1
1809,2,9,2,1
1828,2,9,2,1
1846,2,9,2,1


### **Train/Test split**
We are going to split Train/Test set based on season, as because, F1 data are of Temporal/grouped nature.
___

In [12]:
train_df = model_df[model_df['Season'] <= 2024].copy()
test_df = model_df[model_df['Season'] == 2025].copy()

print("Train:", train_df.shape)
print("Test:", test_df.shape)
print("Train seasons:", sorted(train_df['Season'].unique()))
print("Test seasons:", sorted(test_df['Season'].unique()))

Train: (156810, 33)
Test: (25903, 33)
Train seasons: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Test seasons: [np.int64(2025)]


In [13]:
model_feature_cols = [
    'Driver_enc', 'Team_enc', 'LapNumber', 'Stint',
    'Compound_category_enc', 'TyreLife', 'FreshTyre', 'tyrelife_squared',
    'GridPosition', 'AirTemp', 'Humidity', 'Pressure', 'Rainfall', 
    'TrackTemp', 'WindDirection', 'WindSpeed', 'EventName_enc',
    'is_out_lap', 'fuel_load_proxy', 'track_evolution_proxy',
    'prevstatus_green', 'prevstatus_yellow', 'prevstatus_safety_car',
    'prevstatus_red_flag', 'prevstatus_vsc', 'prevstatus_vsc_ending', 
    'prevstatus_unknown'
]

In [14]:
X_train = train_df[model_feature_cols]
y_train = train_df[target_col]

In [15]:
X_test = test_df[model_feature_cols]
y_test = test_df[target_col]

In [16]:
print(len(model_feature_cols), "final model features")
print(X_train.shape, X_test.shape)
X_train.dtypes

27 final model features
(156810, 27) (25903, 27)


Driver_enc                 int64
Team_enc                   int64
LapNumber                float64
Stint                    float64
Compound_category_enc      int64
TyreLife                 float64
FreshTyre                   bool
tyrelife_squared         float64
GridPosition             float64
AirTemp                  float64
Humidity                 float64
Pressure                 float64
Rainfall                 float64
TrackTemp                float64
WindDirection            float64
WindSpeed                float64
EventName_enc              int64
is_out_lap                  bool
fuel_load_proxy          float64
track_evolution_proxy    float64
prevstatus_green            bool
prevstatus_yellow           bool
prevstatus_safety_car       bool
prevstatus_red_flag         bool
prevstatus_vsc              bool
prevstatus_vsc_ending       bool
prevstatus_unknown          bool
dtype: object

### **XGBoost BASELINE**
___

In [28]:
!pip install xgboost

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   - -------------------------------------- 2.1/48.9 MB 12.0 MB/s eta 0:00:04
   --- ------------------------------------ 4.7/48.9 MB 11.9 MB/s eta 0:00:04
   ----- ---------------------------------- 7.3/48.9 MB 11.9 MB/s eta 0:00:04
   ------- -------------------------------- 9.7/48.9 MB 11.8 MB/s eta 0:00:04
   ---------- ----------------------------- 12.3/48.9 MB 11.7 MB/s eta 0:00:04
   ------------ --------------------------- 14.9/48.9 MB 11.7 MB/s eta 0:00:03
   -------------- ------------------------- 17.3/48.9 MB 11.7 MB/s eta 0:00:03
   ---------------- ----------------------- 19.9/48.9 MB 11.8 MB/s eta 0:00:03
   ------------------ --------------------- 22.5/48.9 MB 11.7 MB/s eta 0:00:03
   -------------------- ------------------- 24.9/48.9 MB 11.7 MB/s eta 0:00:03
   ---------------------- ----------------- 27.5/48.9 MB 11.7 MB/s


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


#### **Training** 

In [17]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import time

model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

start = time.time()
model.fit(X_train, y_train)
print(f"Training time: {time.time() - start:.1f}s")

#Hyperparameters values are default-ish filled, for baseline model.

Training time: 6.4s


#### **Evaluating**

In [18]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

print(f"MAE:  {mae:.3f} seconds")
print(f"RMSE: {rmse:.3f} seconds")
print(f"R2:   {r2:.3f}")

MAE:  3.997 seconds
RMSE: 6.156 seconds
R2:   0.801


## Baseline Model Results (XGBoost, v1)

First baseline trained on the forward-looking feature set (27 features, 
pre-lap known info only). Two data issues were caught and fixed before this 
run:
- Red-flag-affected laps excluded (using `status_red_flag`), since those laps 
  have corrupted LapTime values (stoppage duration baked into the lap, as 
  documented in notebook 06) that don't represent real racing conditions.
- `is_out_lap` corrected to never be True for Lap 1, since FastF1 marks 
  formation/rolling-start laps with a PitOutTime even though Lap 1 isn't a 
  real pit-out in the driving sense.

Results on the 2025 holdout test set:

- MAE: 3.997 seconds
- RMSE: 6.156 seconds
- R2: 0.801

### What this means
On average, the model's lap time prediction is off by about 4 seconds. 
Typical F1 lap times range from ~70s (Monaco) to ~95-100s (Spa, Silverstone), 
so this is roughly a 4-5% error relative to a typical lap. R2 of 0.801 means 
the model explains about 80% of the variance in lap times using only pre-lap 
information, a strong result for a forward-looking model with no in-lap 
telemetry access.

### Is this good or bad?
Solid baseline for a first pass, no tuning done yet. Still room to tighten 
the error down for it to be reliable in tight strategy-margin scenarios 
(undercut/overcut windows are often 1-2 seconds).

### What's next
1. Residual analysis to find where the model is most wrong (done below).
2. Hyperparameter tuning to bring the overall error down further.
3. Consider adding remaining high-value features parked earlier (rolling 
   driver pace, gap to leader) if feasible.

### Target for a strong model
Aiming to bring MAE down to roughly 1.5-2.5 seconds, a strong, honest result 
for a forward-looking model usable in live strategy simulation.

___
### **Feature Importance**

In [19]:
import pandas as pd

importance_df = pd.DataFrame({
    'feature': model_feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

importance_df

,feature,importance
22,prevstatus_safety_car,0.165663
16,EventName_enc,0.124425
26,prevstatus_unknown,0.110647
11,Pressure,0.091413
17,is_out_lap,0.063538
20,prevstatus_green,0.054862
19,track_evolution_proxy,0.053486
18,fuel_load_proxy,0.048145
2,LapNumber,0.036008
5,TyreLife,0.033684


### **Residual Analysis**
___

In [20]:
test_df_analysis = test_df.copy()
test_df_analysis['predicted'] = y_pred
test_df_analysis['abs_error'] = (test_df_analysis[target_col] - test_df_analysis['predicted']).abs()

print("Worst 15 predictions:")
test_df_analysis.sort_values('abs_error', ascending=False)[
    ['EventName', 'Driver', 'LapNumber', target_col, 'predicted', 'abs_error',
     'prevstatus_safety_car', 'prevstatus_vsc', 'prevstatus_red_flag', 'is_out_lap']
].head(15)

Worst 15 predictions:


,EventName,Driver,LapNumber,LapTime_seconds,predicted,abs_error,prevstatus_safety_car,prevstatus_vsc,prevstatus_red_flag,is_out_lap
457244,Saudi_Arabian_Grand_Prix,TSU,1.0,165.662,99.809364,65.852636,False,False,False,False
417800,Belgian_Grand_Prix,ALO,1.0,218.679,156.993942,61.685058,False,False,False,False
417815,Belgian_Grand_Prix,ANT,1.0,216.399,156.096909,60.302091,False,False,False,False
417798,Belgian_Grand_Prix,HAM,1.0,213.712,156.856705,56.855295,False,False,False,False
417810,Belgian_Grand_Prix,SAI,1.0,211.318,156.285355,55.032645,False,False,False,False
417811,Belgian_Grand_Prix,STR,1.0,209.116,156.955826,52.160174,False,False,False,False
417812,Belgian_Grand_Prix,HUL,1.0,206.488,154.359604,52.128396,False,False,False,False
417806,Belgian_Grand_Prix,GAS,1.0,204.447,152.330688,52.116312,False,False,False,False
417808,Belgian_Grand_Prix,OCO,1.0,201.517,149.970627,51.546373,False,False,False,False
417805,Belgian_Grand_Prix,COL,1.0,207.440,157.023224,50.416776,False,False,False,False


In [21]:
check_spa = df_race[
    (df_race['EventName'] == 'Belgian_Grand_Prix') & 
    (df_race['Season'] == 2025) & 
    (df_race['LapNumber'] == 1)
][['Driver', 'LapNumber', 'LapTime_seconds', 'TrackStatus', 
   'status_safety_car', 'status_vsc', 'is_out_lap', 'PitOutTime']]

check_spa

,Driver,LapNumber,LapTime_seconds,TrackStatus,status_safety_car,status_vsc,is_out_lap,PitOutTime
417804,ALB,1.0,191.296,14,True,False,False,0 days 02:14:56.100000
417800,ALO,1.0,218.679,14,True,False,False,0 days 02:15:57.928000
417815,ANT,1.0,216.399,14,True,False,False,0 days 02:15:56.303000
417802,BEA,1.0,201.972,14,True,False,False,0 days 02:15:27.631000
417809,BOR,1.0,199.512,14,True,False,False,0 days 02:15:17.861000
417805,COL,1.0,207.440,14,True,False,False,0 days 02:15:39.242000
417806,GAS,1.0,204.447,14,True,False,False,0 days 02:15:31.943000
417801,HAD,1.0,196.869,14,True,False,False,0 days 02:15:09.438000
417798,HAM,1.0,213.712,14,True,False,False,0 days 02:15:53.006000
417812,HUL,1.0,206.488,14,True,False,False,0 days 02:15:36.143000


## Model Limitation: Safety-Car-Start Races (Signal Gap)

Residual analysis on the worst predictions surfaced a consistent pattern: 
almost the entire top-15 worst predictions were Belgian GP 2025, Lap 1. 
Actual lap times were genuinely long (~185-220 seconds, not corrupted), 
because the race started behind the safety car due to wet conditions 
(confirmed via `status_safety_car = True` on the raw Lap 1 rows for every 
driver that race). This is real data, not a data quality bug like the red 
flag issue.

The reason the model can't catch this: `prevstatus_safety_car` (the shifted, 
pre-lap-known version of track status used as a feature) defaults to 
green/False for every driver's Lap 1 in every race, since there's no "lap 0" 
to shift from. This default is correct most of the time (races normally 
start under green flag), but wrong for the handful of races per season that 
start under safety car or behind a rolling start due to weather. The model 
has no feature that tells it "this specific race is starting under SC," so 
it falls back on the general Lap 1 pattern and underpredicts badly for these 
specific races.

This was not fixed, deliberately. Excluding these rows would misrepresent 
model performance (the data isn't corrupted, the model is just missing a 
signal), and there's no clean pre-lap data source currently available in 
this pipeline to flag "this race is starting under SC/rolling start" ahead 
of time. Fixing it properly would need an external signal per race (e.g. 
official race start classification), which is out of scope for v1.

Impact: a small number of races per season (rain-affected starts) will have 
their Lap 1 systematically underpredicted by this model. This is documented 
as a known v1 limitation, not silently hidden. Confirmed via the error 
breakdown by track status:

- Laps following a safety car: mean abs error ~10.8s (vs ~3.7s otherwise)
- Laps following a VSC: mean abs error ~9.5s (vs ~3.9s otherwise)
- Out-laps: mean abs error ~7.4s (vs ~3.9s otherwise)

These categories are inherently harder to predict from pre-lap information 
alone, and contribute disproportionately to RMSE (which penalizes large 
errors more heavily than MAE). Hyperparameter tuning is expected to improve 
overall MAE and RMSE on the bulk of "normal" laps, but won't resolve this 
specific signal gap since it's a missing-feature problem, not a model-fit 
problem.

### **HyperParameter Tuning**
___

**RandomizedSearchCV**

In [92]:
from sklearn.model_selection import RandomizedSearchCV, GroupKFold
from xgboost import XGBRegressor

param_dist = {
    'n_estimators': [200, 300, 400, 500, 600],
    'max_depth': [4, 5, 6, 7, 8, 9],
    'learning_rate': [0.01, 0.03, 0.05, 0.07, 0.1],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5, 7],
    'gamma': [0, 0.1, 0.2, 0.3]
}

base_model = XGBRegressor(random_state=42, n_jobs=-1)

groups = train_df['Season'].astype(str) + '_' + train_df['EventName'].astype(str)

group_kfold = GroupKFold(n_splits=5)

random_search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_dist,
    n_iter=50,
    scoring='neg_mean_absolute_error',
    cv=group_kfold.split(X_train, y_train, groups=groups),
    verbose=2,
    random_state=42,
    n_jobs=-1
)

In [93]:
random_search.fit(X_train, y_train)

print("Best params:", random_search.best_params_)
print("Best CV MAE:", -random_search.best_score_)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best params: {'subsample': 0.6, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 7, 'learning_rate': 0.07, 'gamma': 0.3, 'colsample_bytree': 1.0}
Best CV MAE: 5.348975754694489


### **Model With best Params**
___

In [94]:
best_model = XGBRegressor(
    subsample=0.6,
    n_estimators=200,
    min_child_weight=5,
    max_depth=7,
    learning_rate=0.07,
    gamma=0.3,
    colsample_bytree=1.0,
    random_state=42,
    n_jobs=-1
)

best_model.fit(X_train, y_train)
y_pred_tuned = best_model.predict(X_test)

mae_tuned = mean_absolute_error(y_test, y_pred_tuned)
rmse_tuned = mean_squared_error(y_test, y_pred_tuned) ** 0.5
r2_tuned = r2_score(y_test, y_pred_tuned)

print(f"MAE:  {mae_tuned:.3f} seconds")
print(f"RMSE: {rmse_tuned:.3f} seconds")
print(f"R2:   {r2_tuned:.3f}")

MAE:  4.278 seconds
RMSE: 6.706 seconds
R2:   0.764


## Hyperparameter Tuning Attempt

Ran RandomizedSearchCV (50 candidates, GroupKFold with 5 splits grouped by 
Season+EventName to avoid the same temporal leakage issue as the train/test 
split) to tune XGBoost hyperparameters.

Best params found (by CV MAE): `subsample=0.6, n_estimators=200, 
min_child_weight=5, max_depth=7, learning_rate=0.07, gamma=0.3, 
colsample_bytree=1.0`

Result on the 2025 test set with these params:
- MAE: 4.278 seconds (vs 3.997 baseline)
- RMSE: 6.706 seconds (vs 6.156 baseline)
- R2: 0.764 (vs 0.801 baseline)

**This is worse than the untuned baseline, not better.**

### Why this happened
The best params from CV were heavily regularized (low subsample, added gamma, 
higher min_child_weight), because GroupKFold trains on a smaller slice of 
data per fold (4/5 of training data, held out by race) and CV likely favored 
regularization to avoid overfitting on those smaller folds. When applied to 
the full training set (all 7 seasons, no holdout), those same regularization 
settings caused the model to underfit slightly, giving up real signal that 
the untuned defaults were capturing.

### Decision
Keeping the original untuned baseline model (n_estimators=300, max_depth=6, 
learning_rate=0.05,

## **Feature Engineering**
**As per guide of Kimi.ai**
___

**Tire degradation features**

In [23]:
df_race_clean['stint_lap_number'] = df_race_clean['TyreLife'] + 1
#Tyrelife_sqrt/log1p - to capture non-linearity of tyre degradation.
df_race_clean['tyrelife_sqrt'] = np.sqrt(df_race_clean['TyreLife'])
df_race_clean['tyrelife_log1p'] = np.log1p(df_race_clean['TyreLife'])

**Compound interactions (one-hot approach)**

 - Compound category is of only 3-4 types, OHE is generally safe here due to low cardinality.
 - One-hot x TyreLife gives the model degradation curve explicitly
 - Compound x TrackTemp gives the interactionof tyre compund and track temperature.

In [24]:
compound_dummies = pd.get_dummies(df_race_clean['Compound_category'], prefix='compound')
df_race_clean = pd.concat([df_race_clean, compound_dummies], axis=1)

for col in compound_dummies.columns:
    df_race_clean[f'{col}_x_tyrelife'] = df_race_clean[col] * df_race_clean['TyreLife']
    df_race_clean[f'{col}_x_tracktemp'] = df_race_clean[col] * df_race_clean['TrackTemp']

df_race_clean.filter(like='compound_').head()

,compound_tyrelife,compound_HARD,compound_INTERMEDIATE,compound_MEDIUM,compound_SOFT,compound_UNKNOWN,compound_WET,compound_HARD_x_tyrelife,compound_HARD_x_tracktemp,compound_INTERMEDIATE_x_tyrelife,compound_INTERMEDIATE_x_tracktemp,compound_MEDIUM_x_tyrelife,compound_MEDIUM_x_tracktemp,compound_SOFT_x_tyrelife,compound_SOFT_x_tracktemp,compound_UNKNOWN_x_tyrelife,compound_UNKNOWN_x_tracktemp,compound_WET_x_tyrelife,compound_WET_x_tracktemp
1709,MEDIUM_1.0,False,False,True,False,False,False,0.0,0.0,0.0,0.0,1.0,34.7,0.0,0.0,0.0,0.0,0.0,0.0
1733,MEDIUM_2.0,False,False,True,False,False,False,0.0,0.0,0.0,0.0,2.0,34.1,0.0,0.0,0.0,0.0,0.0,0.0
1752,MEDIUM_3.0,False,False,True,False,False,False,0.0,0.0,0.0,0.0,3.0,34.6,0.0,0.0,0.0,0.0,0.0,0.0
1771,MEDIUM_4.0,False,False,True,False,False,False,0.0,0.0,0.0,0.0,4.0,34.6,0.0,0.0,0.0,0.0,0.0,0.0
1790,MEDIUM_5.0,False,False,True,False,False,False,0.0,0.0,0.0,0.0,5.0,34.7,0.0,0.0,0.0,0.0,0.0,0.0


**Track evolution, refined (green laps based)**

- Our old `track_evolution_proxy` was simply the raw `LapNumber`, which is crude because rubber isn't laid down during Safety Car laps (as the car moves slowly), and out-laps don't contribute significant rubber either.
- cumulative_green_laps` counts only those laps run under green flag conditions (utilizing our existing `prevstatus_green` metric); consequently, SC/VSC periods are excluded, providing a more accurate signal for track rubbering-in.
- green_laps_sqrt` captures non-linear saturation (where rubbering-in happens rapidly during the first 10–20 green laps before plateauing).

In [32]:
df_race_clean = df_race_clean.sort_values(group_keys + ['LapNumber'])

df_race_clean['cumulative_green_laps'] = (
    df_race_clean.groupby(group_keys)['prevstatus_green']
    .transform(lambda x: x.shift(0).astype(int).cumsum())
)

df_race_clean['green_laps_sqrt'] = np.sqrt(df_race_clean['cumulative_green_laps'])

prev_sc_shifted = df_race_clean.groupby(group_keys)['prevstatus_safety_car'].shift(1)
prev_sc_filled = np.where(prev_sc_shifted.isna(), False, prev_sc_shifted).astype(bool)

df_race_clean['is_first_green_after_sc'] = (
    df_race_clean['prevstatus_green'] & prev_sc_filled
)

**Driver rolling pace (careful with leakage)**

- `clean_mask` identifies laps that are "reference-worthy" for calculating pace—specifically, green-flag laps (excluding out-laps and red-flag periods).
- This is crucial because including Safety Car (SC) lap times in a rolling average would distort the calculation of a driver's "real pace," given that SC laps are significantly slower.
- The `.rolling(3).median().shift(1)` trick works by first calculating the rolling median (which includes the current lap) and then shifting the entire series down by one position using `.shift(1)`.
- The result is that row N receives the value calculated for row N-1—meaning it reflects the median of the three laps *prior* to the current one, effectively excluding the current lap's own time.
- These values ​​were calculated only for the `clean_laps` subset and then assigned back to `df_race_clean` at their original indices.
- `.ffill()` (forward fill) is necessary because non-clean laps (such as Safety Car laps) would have `NaN` pace values ​​in this calculation; we filled them using the pace from the last known clean lap so that every row would reflect a valid "driver's recent form."
- `driver_pace_index = current form / best form`; if the value exceeds 1.0, it means the driver is currently running slower than their best pace.

In [33]:
clean_mask = (
    df_race_clean['status_green'] & 
    (~df_race_clean['is_out_lap']) & 
    (df_race_clean['status_red_flag'] == False)
)

clean_laps = df_race_clean[clean_mask].sort_values(group_keys + ['LapNumber']).copy()

clean_laps['driver_prev_lap_time'] = clean_laps.groupby(group_keys)['LapTime_seconds'].shift(1)
clean_laps['driver_rolling_median_3'] = (
    clean_laps.groupby(group_keys)['LapTime_seconds']
    .transform(lambda x: x.rolling(3, min_periods=1).median().shift(1))
)
clean_laps['driver_best_lap_so_far'] = (
    clean_laps.groupby(group_keys)['LapTime_seconds']
    .transform(lambda x: x.expanding().min().shift(1))
)

pace_cols = ['driver_prev_lap_time', 'driver_rolling_median_3', 'driver_best_lap_so_far']
df_race_clean.loc[clean_laps.index, pace_cols] = clean_laps[pace_cols]

df_race_clean[pace_cols] = (
    df_race_clean.groupby(group_keys)[pace_cols].transform(lambda x: x.ffill())
)

df_race_clean['driver_pace_index'] = (
    df_race_clean['driver_rolling_median_3'] / df_race_clean['driver_best_lap_so_far']
)

**Position, race progress, weather deltas, remaining interactions**

- `position_prev_lap` represents the position from the previous lap (the 'Position' column itself was excluded because it reflects the end-state of the current lap, whereas using `shift(1)` safely captures the value from the previous lap).
- Since there is no previous lap for Lap 1, `GridPosition` is used as a reasonable default (representing the starting position).
- `grid_delta` indicates the positions a driver has gained or lost relative to their grid position.
- `race_progress_pct` normalizes the `LapNumber` to a 0–1 scale, making short and long races (or short vs. long circuits) comparable.
- `rainfall_accumulated` tracks the total rainfall during the session so far (even if current rainfall is zero, a previously wet track still affects grip).
- The remaining interaction terms (Stint×TyreLife, GridPosition×progress, AirTemp×TrackTemp) are based on Kimi's suggestion and involve direct multiplication.

In [34]:
df_race_clean['position_prev_lap'] = (
    df_race_clean.groupby(group_keys)['Position'].shift(1)
)
df_race_clean['position_prev_lap'] = df_race_clean['position_prev_lap'].fillna(df_race_clean['GridPosition'])

df_race_clean['grid_delta'] = df_race_clean['position_prev_lap'] - df_race_clean['GridPosition']
df_race_clean['is_race_leader'] = (df_race_clean['position_prev_lap'] == 1)

df_race_clean['race_progress_pct'] = df_race_clean['LapNumber'] / df_race_clean['total_laps_est']
df_race_clean['fuel_load_pct'] = df_race_clean['laps_remaining_est'] / df_race_clean['total_laps_est']

df_race_clean['rainfall_accumulated'] = (
    df_race_clean.groupby(group_keys)['Rainfall'].transform('cumsum')
)
df_race_clean['track_temp_minus_airtemp'] = df_race_clean['TrackTemp'] - df_race_clean['AirTemp']

df_race_clean['stint_x_tyrelife'] = df_race_clean['Stint'] * df_race_clean['TyreLife']
df_race_clean['gridpos_x_progress'] = df_race_clean['GridPosition'] * df_race_clean['race_progress_pct']
df_race_clean['airtemp_x_tracktemp'] = df_race_clean['AirTemp'] * df_race_clean['TrackTemp']

### **REBUILD - BRICK BY BRICK**
___

In [35]:
feature_cols = [
    'Driver', 'Team',
    'LapNumber', 'Stint',
    'Compound_category', 'TyreLife', 'FreshTyre', 'tyrelife_squared',
    'GridPosition',
    'AirTemp', 'Humidity', 'Pressure', 'Rainfall', 'TrackTemp', 
    'WindDirection', 'WindSpeed',
    'EventName', 'Season',
    'is_out_lap', 'fuel_load_proxy', 'track_evolution_proxy',
    'prevstatus_green', 'prevstatus_yellow', 'prevstatus_safety_car',
    'prevstatus_red_flag', 'prevstatus_vsc', 'prevstatus_vsc_ending', 
    'prevstatus_unknown',
    
    # new tier 1 features
    'stint_lap_number', 'tyrelife_sqrt', 'tyrelife_log1p',
    'cumulative_green_laps', 'green_laps_sqrt', 'is_first_green_after_sc',
    'driver_prev_lap_time', 'driver_rolling_median_3', 
    'driver_best_lap_so_far', 'driver_pace_index',
    'position_prev_lap', 'grid_delta', 'is_race_leader',
    'race_progress_pct', 'fuel_load_pct',
    'rainfall_accumulated', 'track_temp_minus_airtemp',
    'stint_x_tyrelife', 'gridpos_x_progress', 'airtemp_x_tracktemp'
]

compound_interaction_cols = [c for c in df_race_clean.columns 
                              if c.startswith('compound_') and ('_x_tyrelife' in c or '_x_tracktemp' in c)]
compound_base_cols = [c for c in df_race_clean.columns 
                       if c.startswith('compound_') and c not in compound_interaction_cols]

feature_cols = feature_cols + compound_interaction_cols + compound_base_cols

print(len(feature_cols), "features")

67 features


In [36]:
print("Red flag rows still present:", (df_race_clean['status_red_flag'] == True).sum())

Red flag rows still present: 0


In [37]:
model_df = df_race_clean[feature_cols + [target_col]].copy()

before = len(model_df)
model_df = model_df.dropna()
after = len(model_df)

print(f"Dropped {before - after} rows ({(before-after)/before*100:.2f}%)")
print(model_df.shape)

Dropped 10936 rows (5.82%)
(177120, 68)


In [38]:
categorical_cols = ['Driver', 'Team', 'Compound_category', 'EventName']

encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    model_df[col + '_enc'] = le.fit_transform(model_df[col])
    encoders[col] = le

train_df = model_df[model_df['Season'] <= 2024].copy()
test_df = model_df[model_df['Season'] == 2025].copy()

print("Train:", train_df.shape)
print("Test:", test_df.shape)

Train: (151829, 72)
Test: (25291, 72)


In [39]:
model_feature_cols = [c for c in feature_cols if c not in categorical_cols] + \
                      [c + '_enc' for c in categorical_cols]

X_train = train_df[model_feature_cols]
y_train = train_df[target_col]
X_test = test_df[model_feature_cols]
y_test = test_df[target_col]

print(len(model_feature_cols), "final model features")
print(X_train.dtypes.value_counts())

67 final model features
float64    43
bool       17
int64       6
object      1
Name: count, dtype: int64


In [40]:
X_train.dtypes[X_train.dtypes == 'object']

compound_tyrelife    object
dtype: object

In [44]:
feature_cols = [c for c in feature_cols if c != 'compound_tyrelife']

In [45]:
model_df = df_race_clean[feature_cols + [target_col]].copy()

before = len(model_df)
model_df = model_df.dropna()
after = len(model_df)

print(f"Dropped {before - after} rows ({(before-after)/before*100:.2f}%)")
print(model_df.shape)

Dropped 10936 rows (5.82%)
(177120, 67)


In [48]:
model_feature_cols = [c for c in feature_cols if c not in categorical_cols] + \
                      [c + '_enc' for c in categorical_cols]

X_train = train_df[model_feature_cols]
y_train = train_df[target_col]
X_test = test_df[model_feature_cols]
y_test = test_df[target_col]

print(len(model_feature_cols), "final model features")
print(X_train.dtypes.value_counts())

66 final model features
float64    43
bool       17
int64       6
Name: count, dtype: int64


In [49]:
model_v2 = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model_v2.fit(X_train, y_train)
y_pred_v2 = model_v2.predict(X_test)

mae_v2 = mean_absolute_error(y_test, y_pred_v2)
rmse_v2 = mean_squared_error(y_test, y_pred_v2) ** 0.5
r2_v2 = r2_score(y_test, y_pred_v2)

print(f"MAE:  {mae_v2:.3f} seconds")
print(f"RMSE: {rmse_v2:.3f} seconds")
print(f"R2:   {r2_v2:.3f}")

MAE:  1.720 seconds
RMSE: 4.199 seconds
R2:   0.896


**GOOD !!!**

**Feature Importance**

In [52]:
importance_df_v2 = pd.DataFrame({
    'feature': model_feature_cols,
    'importance': model_v2.feature_importances_
}).sort_values('importance', ascending=False)

importance_df_v2.head(20)

,feature,importance
30,driver_prev_lap_time,0.265895
32,driver_best_lap_so_far,0.257218
19,prevstatus_safety_car,0.078719
17,prevstatus_green,0.059683
14,is_out_lap,0.059518
31,driver_rolling_median_3,0.050374
29,is_first_green_after_sc,0.023551
4,tyrelife_squared,0.019152
21,prevstatus_vsc,0.013147
15,fuel_load_proxy,0.009783


**Residual Analysis**

In [54]:
test_df_v2 = test_df.copy()
test_df_v2['predicted'] = y_pred_v2
test_df_v2['abs_error'] = (test_df_v2[target_col] - test_df_v2['predicted']).abs()

print("Worst 15 predictions:")
test_df_v2.sort_values('abs_error', ascending=False)[
    ['EventName', 'Driver', 'LapNumber', target_col, 'predicted', 'abs_error']
].head(15)

Worst 15 predictions:


,EventName,Driver,LapNumber,LapTime_seconds,predicted,abs_error
454109,Qatar_Grand_Prix,GAS,7.0,142.022,88.081100,53.940900
408654,Australian_Grand_Prix,OCO,46.0,141.745,94.990860,46.754140
424891,Canadian_Grand_Prix,STR,66.0,123.386,78.928993,44.457007
429618,Dutch_Grand_Prix,LAW,27.0,124.543,81.808617,42.734383
429619,Dutch_Grand_Prix,SAI,27.0,123.374,82.064331,41.309669
424890,Canadian_Grand_Prix,HAD,66.0,121.476,81.204536,40.271464
424884,Canadian_Grand_Prix,BEA,66.0,118.863,78.943237,39.919763
408445,Australian_Grand_Prix,BOR,33.0,130.326,91.067596,39.258404
448489,Miami_Grand_Prix,LEC,29.0,131.178,92.970894,38.207106
408663,Australian_Grand_Prix,BEA,46.0,125.498,88.101082,37.396918


In [55]:
for col in ['prevstatus_safety_car', 'prevstatus_vsc', 'is_out_lap']:
    print(f"\n{col}:")
    print(test_df_v2.groupby(col)['abs_error'].agg(['mean', 'count']))


prevstatus_safety_car:
                           mean  count
prevstatus_safety_car                 
False                  1.369193  24168
True                   9.274873   1123

prevstatus_vsc:
                    mean  count
prevstatus_vsc                 
False           1.612620  24851
True            7.798009    440

is_out_lap:
                mean  count
is_out_lap                 
False       1.608776  24563
True        5.480724    728


## v2 Results: Feature Engineering Impact

Retrained the model with the Tier 1 feature engineering additions (driver 
rolling pace, track evolution refinement, tire degradation curves, position/
strategy context, weather deltas, and key interaction terms), same untuned 
baseline hyperparameters as before for a fair comparison.

Results on the 2025 holdout test set:

- MAE: 1.720 seconds (down from 3.997)
- RMSE: 4.199 seconds (down from 6.156)
- R2: 0.896 (up from 0.801)

### What changed
This is a major jump, MAE dropped by more than half. Feature importance 
confirms why: `driver_prev_lap_time` and `driver_best_lap_so_far` alone 
account for ~52% of total importance. Driver recent form was the single 
biggest missing signal in v1, adding it (via careful leakage-safe rolling 
features, using only prior clean laps) gave the model the context it needed 
to actually predict pace instead of relying mostly on static conditions.

This result is now within the target range set at the start of this notebook 
(1.5-2.5s MAE for a forward-looking model usable in live strategy simulation).

### Remaining weak spot
Track-status-dependent laps are still predicted much worse than normal laps:

- Laps under safety car: mean abs error ~9.3s (vs ~1.4s for

___
**New Features for Edge Case Handling**

In [59]:
group_keys_local = ['Season', 'EventName', 'SessionName', 'Driver']

def consecutive_count(series):
    result = []
    count = 0
    for val in series:
        if val:
            count += 1
        else:
            count = 0
        result.append(count)
    return result

def sc_deployment_transform(x):
    shifted = x.shift(1)
    shifted_filled = np.where(shifted.isna(), False, shifted).astype(bool)
    is_new_deployment = x.values & ~shifted_filled
    return pd.Series(is_new_deployment, index=x.index).cumsum()

df_race_clean = df_race_clean.sort_values(group_keys_local + ['LapNumber'])

df_race_clean['laps_since_sc_started'] = (
    df_race_clean.groupby(group_keys_local)['prevstatus_safety_car']
    .transform(lambda x: consecutive_count(x))
)

df_race_clean['laps_since_vsc_started'] = (
    df_race_clean.groupby(group_keys_local)['prevstatus_vsc']
    .transform(lambda x: consecutive_count(x))
)

df_race_clean['sc_deployment_count'] = (
    df_race_clean.groupby(group_keys_local)['prevstatus_safety_car']
    .transform(sc_deployment_transform)
)

df_race_clean[['laps_since_sc_started', 'laps_since_vsc_started', 'sc_deployment_count']].describe()

,laps_since_sc_started,laps_since_vsc_started,sc_deployment_count
count,188056.000000,188056.000000,188056.000000
mean,0.169933,0.028183,0.440725
std,0.841729,0.240096,0.598915
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000
75%,0.000000,0.000000,1.000000
max,12.000000,7.000000,4.000000


In [60]:
feature_cols = feature_cols + ['laps_since_sc_started', 'laps_since_vsc_started', 'sc_deployment_count']
print(len(feature_cols), "features")

69 features


In [61]:
model_df = df_race_clean[feature_cols + [target_col]].copy()

before = len(model_df)
model_df = model_df.dropna()
after = len(model_df)

print(f"Dropped {before - after} rows ({(before-after)/before*100:.2f}%)")
print(model_df.shape)

Dropped 10936 rows (5.82%)
(177120, 70)


In [62]:
categorical_cols = ['Driver', 'Team', 'Compound_category', 'EventName']

encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    model_df[col + '_enc'] = le.fit_transform(model_df[col])
    encoders[col] = le

train_df = model_df[model_df['Season'] <= 2024].copy()
test_df = model_df[model_df['Season'] == 2025].copy()

print("Train:", train_df.shape)
print("Test:", test_df.shape)

Train: (151829, 74)
Test: (25291, 74)


In [63]:
model_feature_cols = [c for c in feature_cols if c not in categorical_cols] + \
                      [c + '_enc' for c in categorical_cols]

X_train = train_df[model_feature_cols]
y_train = train_df[target_col]
X_test = test_df[model_feature_cols]
y_test = test_df[target_col]

print(len(model_feature_cols), "final model features")
print(X_train.dtypes.value_counts())

69 final model features
float64    43
bool       17
int64       9
Name: count, dtype: int64


**Model v3 training**
___

In [65]:
model_v3 = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model_v3.fit(X_train, y_train)
y_pred_v3 = model_v3.predict(X_test)

mae_v3 = mean_absolute_error(y_test, y_pred_v3)
rmse_v3 = mean_squared_error(y_test, y_pred_v3) ** 0.5
r2_v3 = r2_score(y_test, y_pred_v3)

print(f"MAE:  {mae_v3:.3f} seconds")
print(f"RMSE: {rmse_v3:.3f} seconds")
print(f"R2:   {r2_v3:.3f}")

MAE:  1.727 seconds
RMSE: 4.225 seconds
R2:   0.895


In [66]:
print("Red flag check:", (df_race_clean['status_red_flag'] == True).sum())
print("Any nulls in new features:", df_race_clean[['laps_since_sc_started', 'laps_since_vsc_started', 'sc_deployment_count']].isna().sum().sum())

Red flag check: 0
Any nulls in new features: 0


In [67]:
test_df_v3 = test_df.copy()
test_df_v3['predicted'] = y_pred_v3
test_df_v3['abs_error'] = (test_df_v3[target_col] - test_df_v3['predicted']).abs()

for col in ['prevstatus_safety_car', 'prevstatus_vsc', 'is_out_lap']:
    print(f"\n{col}:")
    print(test_df_v3.groupby(col)['abs_error'].agg(['mean', 'count']))


prevstatus_safety_car:
                           mean  count
prevstatus_safety_car                 
False                  1.353273  24168
True                   9.775687   1123

prevstatus_vsc:
                    mean  count
prevstatus_vsc                 
False           1.621508  24851
True            7.699795    440

is_out_lap:
                mean  count
is_out_lap                 
False       1.624855  24563
True        5.182273    728


## Attempt: Targeted SC/VSC Features (Did Not Help)

Added three features aimed at improving prediction during safety car / VSC 
periods: `laps_since_sc_started`, `laps_since_vsc_started` (consecutive lap 
counters that reset when the status changes), and `sc_deployment_count` 
(cumulative count of distinct SC deployments in the race so far).

Result on the 2025 test set: essentially no change.

- MAE: 1.727 seconds (vs 1.720 in v2)
- RMSE: 4.225 seconds (vs 4.199 in v2)
- R2: 0.895 (vs 0.896 in v2)

Breakdown by track status, compared to v2:

| Condition | v2 mean abs error | v3 mean abs error |
|---|---|---|
| Safety car laps | 9.275s | 9.776s (slightly worse) |
| VSC laps | 7.798s | 7.700s (~same) |
| Out-laps | 5.481s | 5.182s (slightly better) |

No meaningful improvement, SC actually got marginally worse.

### Why this likely didn't work
The hypothesis was that "how long the SC has been out" would help the model 
distinguish early-SC chaos from settled-SC pace. But the real driver of 
lap time variance during SC/VSC periods is probably much more granular than 
that, exact gap to the safety car, exact lap the SC was deployed relative to 
where the driver was on track, how hard the driver was pushing right before 
the deployment. None of this is derivable from the data available in this 
pipeline (no real-time gap/interval telemetry). A coarse "laps since SC 
started" counter doesn't capture that fine-grained variance.

### Decision
Keeping v2 as the final model for lap time prediction. The new SC/VSC 
features are kept in the feature set since they don't hurt anything, but the 
underperformance on SC/VSC/out-laps is accepted as a documented v1 
limitation, consistent with how the earlier SC-start (Belgian GP) signal gap 
was handled. Fixing this properly would require a different, richer data 
source (real-time gap/position telemetry), which is out of scope for this 
version.

### Final v2 model summary
- MAE: 1.720s, RMSE: 4.199s, R2: 0.896 on 2025 holdout
- Well within the target range set at the start (1.5-2.5s MAE)
- Known weak spot: SC/VSC/out-laps predicted with higher error (5-10s), a 
  small fraction of total laps, documented rather than hidden

**HyperParameter Tuning**
___

In [70]:
train_only_df = train_df[train_df['Season'] <= 2022].copy()
val_df = train_df[train_df['Season'].isin([2023, 2024])].copy()

X_train_only = train_only_df[model_feature_cols]
y_train_only = train_only_df[target_col]
X_val = val_df[model_feature_cols]
y_val = val_df[target_col]

print("Train only:", X_train_only.shape)
print("Validation:", X_val.shape)

Train only: (103825, 69)
Validation: (48004, 69)


In [71]:
model_tuned = XGBRegressor(
    n_estimators=5000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50,
    eval_metric='mae'
)

model_tuned.fit(
    X_train_only, y_train_only,
    eval_set=[(X_val, y_val)],
    verbose=True
)

print(f"\nBest iteration: {model_tuned.best_iteration}")

[0]	validation_0-mae:9.72439
[1]	validation_0-mae:9.28372
[2]	validation_0-mae:8.87481
[3]	validation_0-mae:8.48370
[4]	validation_0-mae:8.10453
[5]	validation_0-mae:7.74791
[6]	validation_0-mae:7.40842
[7]	validation_0-mae:7.09077
[8]	validation_0-mae:6.77764
[9]	validation_0-mae:6.48441
[10]	validation_0-mae:6.21944
[11]	validation_0-mae:5.95852
[12]	validation_0-mae:5.71287
[13]	validation_0-mae:5.48804
[14]	validation_0-mae:5.27563
[15]	validation_0-mae:5.07228
[16]	validation_0-mae:4.88200
[17]	validation_0-mae:4.69867
[18]	validation_0-mae:4.52668
[19]	validation_0-mae:4.35729
[20]	validation_0-mae:4.20227
[21]	validation_0-mae:4.05701
[22]	validation_0-mae:3.91209
[23]	validation_0-mae:3.78082
[24]	validation_0-mae:3.65661
[25]	validation_0-mae:3.53442
[26]	validation_0-mae:3.42188
[27]	validation_0-mae:3.31480
[28]	validation_0-mae:3.21307
[29]	validation_0-mae:3.11854
[30]	validation_0-mae:3.02961
[31]	validation_0-mae:2.94672
[32]	validation_0-mae:2.86831
[33]	validation_0-ma

In [72]:
final_n_estimators = model_tuned.best_iteration

model_final = XGBRegressor(
    n_estimators=final_n_estimators,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model_final.fit(X_train, y_train)
y_pred_final = model_final.predict(X_test)

mae_final = mean_absolute_error(y_test, y_pred_final)
rmse_final = mean_squared_error(y_test, y_pred_final) ** 0.5
r2_final = r2_score(y_test, y_pred_final)

print(f"MAE:  {mae_final:.3f} seconds")
print(f"RMSE: {rmse_final:.3f} seconds")
print(f"R2:   {r2_final:.3f}")

MAE:  1.702 seconds
RMSE: 4.276 seconds
R2:   0.892


## Tuning Attempt (Early Stopping) and Final Decision

Tried a more targeted tuning approach this time: used early stopping with a 
validation split (train on 2018-2022, validate on 2023-2024) to let XGBoost 
find the optimal `n_estimators` automatically, while keeping the other 
hyperparameters (learning_rate=0.05, max_depth=6, subsample=0.8, 
colsample_bytree=0.8) fixed at the values that already worked well in v2. 
This was a deliberately conservative approach, the earlier RandomizedSearchCV 
attempt (tuning everything at once with GroupKFold) had made things worse by 
over-regularizing, so this time only one lever was tuned.

Best iteration found: 108 (validation MAE plateaued and started oscillating 
around iteration ~85-140, confirming the model was near its ceiling).

Retrained on full training data (2018-2024) with `n_estimators=108`, 
evaluated on the 2025 test set:

- MAE: 1.702 seconds (vs 1.720 in v2)
- RMSE: 4.276 seconds (vs 4.199 in v2)
- R2: 0.892 (vs 0.896 in v2)

Essentially the same result as v2, marginally better MAE, marginally worse 
RMSE and R2. No meaningful improvement either way. This confirms the model 
has plateaued with the current feature set, further hyperparameter tuning is 
not worth pursuing further.

### Decision: v2 is the final model

Keeping v2 (untuned baseline hyperparameters: n_estimators=300, max_depth=6, 
learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, trained on the full 
Tier 1 feature-engineered dataset) as the final lap time prediction model, 
since it performs equivalently to the tuned version with a simpler setup.

**Final model performance (2025 holdout test set):**
- MAE: 1.720 seconds
- RMSE: 4.199 seconds
- R2: 0.896

This meets the target set at the start of this notebook (1.5-2.5s MAE for a 
forward-looking model usable in live strategy simulation). Known limitation: 
SC/VSC/out-laps are predicted with higher error (5-10s), documented earlier, 
accepted as a v1 limitation due to missing real-time gap/position telemetry.

## **Saving model_v2 and Encoders to the disk**
___

In [76]:
import os
import joblib

save_dir = "../models/lap_time_prediction"
os.makedirs(save_dir, exist_ok=True)

joblib.dump(model_v2, os.path.join(save_dir, "lap_time_model_v2.pkl"))
joblib.dump(encoders, os.path.join(save_dir, "label_encoders.pkl"))
joblib.dump(model_feature_cols, os.path.join(save_dir, "feature_cols.pkl"))

print("Saved to:", save_dir)
print(os.listdir(save_dir))

Saved to: ../models/lap_time_prediction
['feature_cols.pkl', 'label_encoders.pkl', 'lap_time_model_v2.pkl']


## Notebook Summary: Lap Time Prediction Model (Complete)

### What was built
A forward-looking XGBoost model that predicts lap time using only 
information known before a lap starts (driver, tyre state, weather, track 
status, race context). No in-lap telemetry or post-lap flags used, making it 
usable for live strategy simulation in the web app.

### Data
- Source: `fastf1_ml_base.parquet`, filtered to race sessions only
- 188,482 race laps -> 182,841 after initial null cleanup -> 177,120 final 
  modeling rows after feature engineering (some early-race rows dropped due 
  to rolling driver-pace features needing history)
- Train: 2018-2024 seasons, Test: 2025 season (season-based split, not 
  random, to avoid temporal/grouped leakage)

### Key issues found and fixed along the way
1. Q1/Q2/Q3 were 100% null for race rows, dropped from features
2. 2024 Monaco GP red-flag-affected laps were contaminating training data 
   with corrupted LapTime values (~2400-2500s), excluded via `status_red_flag`
3. `is_out_lap` was incorrectly flagging every driver's Lap 1 as an out-lap 
   in races with rolling/SC starts, fixed by excluding Lap 1 from the 
   out-lap definition
4. Track status features use a shift-by-one approximation (previous lap's 
   status), which has a known limitation for races that start under safety 
   car (documented, not fixed, no clean pre-lap signal available for this 
   in the current pipeline)

### Feature engineering (the biggest lever)
Started at 27 features, MAE 4.0s. Added driver rolling pace/form, tire 
degradation curves, track evolution (green-flag-lap based), position/
strategy context, and interaction terms, ending at 69 features, MAE 1.72s. 
Driver recent form (`driver_prev_lap_time`, `driver_best_lap_so_far`) turned 
out to be the single biggest missing signal, accounting for ~52% of feature 
importance in the final model.

An attempt to add SC/VSC-duration-specific features did not improve 
performance and was not kept as an active improvement (features remain in 
the set but don't meaningfully help).

### Hyperparameter tuning
Two attempts, both showed the model was near its practical ceiling for this 
feature set:
1. RandomizedSearchCV with GroupKFold made results worse (over-regularization)
2. Early stopping to auto-tune `n_estimators` gave essentially identical 
   results to the untuned baseline

### Final model
- Untuned XGBoost baseline (n_estimators=300, max_depth=6, learning_rate=0.05, 
  subsample=0.8, colsample_bytree=0.8), trained on the full engineered 
  feature set
- **MAE: 1.720s, RMSE: 4.199s, R2: 0.896** on 2025 holdout
- Saved to `models/lap_time_prediction/`: model file, label encoders, and 
  feature column order (all needed for the web app to reconstruct inputs 
  and predict correctly)

### Known limitations (documented, not silently hidden)
- SC/VSC laps and out-laps predicted with higher error (5-10s vs ~1.4s for 
  normal laps), due to missing real-time gap/position telemetry
- Races starting under safety car/rolling start have their Lap 1 
  systematically underpredicted, since the shift-based track status feature 
  can't know this ahead of time without an external race-start-type signal

### Next steps
- A reusable preprocessing pipeline/function should be built for the next 
  notebook, to avoid the repeated manual cell-by-cell rerun process used here
- Model 2 (tyre degradation) starts in a new notebook: `08_tyre_degradation_model.ipynb`
- Inference-time feature reconstruction logic (deriving the ~69 features 
  from a handful of user-facing inputs) is deferred to the web app 
  development stage, not part of this notebook's scope